In [2]:
%pip install -q transformers peft accelerate

In [19]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [43]:
import random
import torch
import transformers
import peft
import accelerate

CONFIG = {
    "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "dtype": "bfloat16",

    "rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],

    "learning_rate": 2e-4,
    "weight_decay": 0.0,
    "num_epochs": 10,
    "max_length": 256,

    "seed": 42,
    "output_dir": "/content/lora-project/run-001",
}

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

for key, value in CONFIG.items():
    print(f"{key}: {value}")

model_id: TinyLlama/TinyLlama-1.1B-Chat-v1.0
dtype: bfloat16
rank: 8
lora_alpha: 16
lora_dropout: 0.05
target_modules: ['q_proj', 'v_proj']
learning_rate: 0.0002
weight_decay: 0.0
num_epochs: 10
max_length: 256
seed: 42
output_dir: /content/lora-project/run-001


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])

In [24]:
example = {
    "prompt": "Explain LoRA in one sentence.",
    "answer": (
        "LoRA (Low-Rank Adaptation) fine-tunes a model by training "
        "small low-rank matrices while keeping the original weights frozen."
    ),
}
training_messages = [
    {"role": "user", "content": example["prompt"]},
    {"role": "assistant", "content": example["answer"]},
]


<|user|>
Explain LoRA in one sentence.</s>
<|assistant|>
LoRA (Low-Rank Adaptation) fine-tunes a model by training small low-rank matrices while keeping the original weights frozen.</s>



In [ ]:
train_examples = [
    example,

    {
        "prompt": "What stays frozen during standard LoRA training?",
        "answer": (
            "The original pretrained model weights stay frozen. "
            "Only the added LoRA adapter parameters are trained."
        ),
    },
    {
        "prompt": "What are the trainable matrices in a LoRA adapter?",
        "answer": (
            "A LoRA adapter contains two trainable low-rank factors, "
            "A and B. Their product represents a weight update."
        ),
    },
    {
        "prompt": "Is LoRA restricted to the final output layer?",
        "answer": (
            "No. LoRA adapters can be attached to selected linear "
            "layers throughout a model, depending on the configuration."
        ),
    },
    {
        "prompt": "How does LoRA change a linear layer's computation?",
        "answer": (
            "LoRA adds a scaled low-rank branch: "
            "y = W x + (alpha / r) B A x. "
            "W stays frozen while A and B are trained."
        ),
    },
]

print("Training examples:", len(train_examples))

In [46]:
def prepare_example(example):
    user_messages = [
        {"role": "user", "content": example["prompt"]}
    ]

    prompt_text = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_text = tokenizer.apply_chat_template(
        user_messages + [
            {"role": "assistant", "content": example["answer"]}
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    #Full input(question + answer)
    prepared = tokenizer(
        full_text,
        add_special_tokens=False,
        return_tensors="pt",
    )
    #prompt
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    prompt_length = len(prompt_ids)
    input_ids = prepared["input_ids"]

    if input_ids.shape[1] > CONFIG["max_length"]:
        raise ValueError(
            f"Example has {input_ids.shape[1]} tokens; "
            f"max_length is {CONFIG['max_length']}."
        )

    if input_ids[0, :prompt_length].tolist() != prompt_ids:
        raise ValueError("Prompt tokens do not match the full-text prefix.")

    if prompt_length >= input_ids.shape[1]:
        raise ValueError("No answer tokens are available for training.")

    prepared["labels"] = input_ids.clone()
    prepared["labels"][:, :prompt_length] = -100
    return prepared

train_batches = [
    prepare_example(item)
    for item in train_examples
]

for index, batch in enumerate(train_batches, start=1):
    input_count = batch["input_ids"].shape[1]
    target_count = (batch["labels"] != -100).sum().item()

    print(
        f"Example {index}: "
        f"{input_count} input tokens, "
        f"{target_count} supervised tokens, "
        f"device={batch['input_ids'].device}"
    )

Example 1: 58 input tokens, 35 supervised tokens, device=cpu
Example 2: 48 input tokens, 23 supervised tokens, device=cpu
Example 3: 53 input tokens, 26 supervised tokens, device=cpu
Example 4: 50 input tokens, 25 supervised tokens, device=cpu
Example 5: 65 input tokens, 38 supervised tokens, device=cpu


In [47]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
project_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    dtype=getattr(torch, CONFIG["dtype"]),
    device_map=0,
)

project_model.config.use_cache = False

lora_config = LoraConfig(
    r=CONFIG["rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
# Attach LoRA adapters to the selected modules and freeze the base model.
project_model = get_peft_model(project_model, lora_config)

# Initialize the optimizer with only trainable parameters.
project_optimizer = torch.optim.AdamW(
    [param for param in project_model.parameters() if param.requires_grad],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)


project_model.print_trainable_parameters()
print("Optimizer state entries:", len(project_optimizer.state))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
Optimizer state entries: 0


In [52]:
rng = random.Random(CONFIG["seed"])
project_model.train()

for epoch in range(CONFIG["num_epochs"]):
  order = list(range(len(train_batches)))
  rng.shuffle(order)

  total_loss = 0.0
  for index in order:
    batch = {
            key: value.to(project_model.device)
            for key, value in train_batches[index].items()
        }

    project_optimizer.zero_grad()
    loss = project_model(**batch).loss
    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}.")

    loss.backward()
    project_optimizer.step()
    total_loss += loss.item()

  mean_loss = total_loss / len(train_batches)
  print(f"Epoch {epoch + 1:2d} | Mean example loss: {mean_loss:.4f}")

Epoch  1 | Mean example loss: 3.1727
Epoch  2 | Mean example loss: 2.5367
Epoch  3 | Mean example loss: 2.2625
Epoch  4 | Mean example loss: 2.0299
Epoch  5 | Mean example loss: 1.8073
Epoch  6 | Mean example loss: 1.5429
Epoch  7 | Mean example loss: 1.2577
Epoch  8 | Mean example loss: 0.9714
Epoch  9 | Mean example loss: 0.6443
Epoch 10 | Mean example loss: 0.3348


In [55]:
validation_examples = [
    {
        "prompt": (
            "In machine learning, which weights are updated "
            "during LoRA fine-tuning?"
        ),
        "answer": (
            "The LoRA adapter weights are updated while the "
            "original pretrained model weights remain frozen."
        ),
    },
    {
        "prompt": "Why can LoRA reduce the number of trainable parameters?",
        "answer": (
            "LoRA represents a weight update using two low-rank "
            "matrices, which require fewer trainable parameters."
        ),
    },
]

validation_batches = [
    prepare_example(item)
    for item in validation_examples
]


In [56]:
def evaluate_loss(model, batches):
  model.eval()
  total_loss = 0.0

  with torch.inference_mode():
    for cpu_batch in batches:
      batch = {
          key: value.to(model.device)
          for key, value in cpu_batch.items()
      }
      total_loss += model(**batch).loss.item()
  return total_loss / len(batches)
with project_model.disable_adapter():
  base_loss = evaluate_loss(project_model, validation_batches)
lora_loss = evaluate_loss(project_model, validation_batches)

print(f"Base validation loss: {base_loss:.4f}")
print(f"LoRA validation loss: {lora_loss:.4f}")

Base validation loss: 3.5801
LoRA validation loss: 1.6590


In [59]:
def generate_answer(model, tokenizer, prompt):
  model.eval()
  text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
  )
  inputs = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
  ).to(model.device)
  with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

  input_length = inputs["input_ids"].shape[1]
  new_tokens = output[0, input_length:]

  return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [60]:
for example in validation_examples:
    prompt = example["prompt"]

    # Generate with the frozen base model.
    with project_model.disable_adapter():
        base_answer = generate_answer(project_model, tokenizer, prompt)

    # Generate with the trained adapter enabled.
    lora_answer = generate_answer(project_model, tokenizer, prompt)

    print("QUESTION:", prompt)
    print("REFERENCE:", example["answer"])
    print("BASE:", base_answer)
    print("LORA:", lora_answer)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: In machine learning, which weights are updated during LoRA fine-tuning?
REFERENCE: The LoRA adapter weights are updated while the original pretrained model weights remain frozen.
BASE: During LoRA fine-tuning, the weights of the layers in the neural network are updated based on the loss function. The weights of the layers are updated by minimizing the loss function, which is a function that measures the difference between the output of the network and the target label. The weights of the layers
LORA: The low-rank factors are updated.
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: Why can LoRA reduce the number of trainable parameters?
REFERENCE: LoRA represents a weight update using two low-rank matrices, which require fewer trainable parameters.
BASE: LoRA (Low-Rank Alternating Optimization for Reliable Communication) is a technique that reduces the number of trainable parameters in a deep learning model by using a low-rank approximation of the input data. This approach is particularly useful when the input data is sparse or has a high degree
LORA: LoRA reduces the number of trainable parameters by a factor of 16 while maintaining the same weight update.
------------------------------------------------------------


In [61]:
validation_results = []

for item in validation_examples:
    prompt = item["prompt"]

    # Generate both responses with identical settings.
    with project_model.disable_adapter():
        base_answer = generate_answer(project_model, tokenizer, prompt)

    lora_answer = generate_answer(project_model, tokenizer, prompt)

    validation_results.append({
        "prompt": prompt,
        "reference": item["answer"],
        "base_answer": base_answer,
        "lora_answer": lora_answer,
    })

print("Recorded validation examples:", len(validation_results))
print(validation_results[0])

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Recorded validation examples: 2
{'prompt': 'In machine learning, which weights are updated during LoRA fine-tuning?', 'reference': 'The LoRA adapter weights are updated while the original pretrained model weights remain frozen.', 'base_answer': 'During LoRA fine-tuning, the weights of the layers in the neural network are updated based on the loss function. The weights of the layers are updated by minimizing the loss function, which is a function that measures the difference between the output of the network and the target label. The weights of the layers', 'lora_answer': 'The low-rank factors are updated.'}


In [62]:
# Human annotations for the observed validation answers.
human_scores = [
    {"base_score": 0, "lora_score": 1},
    {"base_score": 0, "lora_score": 0},
]

assert len(human_scores) == len(validation_results)

# Attach the scores to their corresponding results.
for result, scores in zip(validation_results, human_scores):
    result.update(scores)

base_mean = sum(
    result["base_score"] for result in validation_results
) / len(validation_results)

lora_mean = sum(
    result["lora_score"] for result in validation_results
) / len(validation_results)

print(f"Base mean score: {base_mean:.2f} / 2")
print(f"LoRA mean score: {lora_mean:.2f} / 2")

Base mean score: 0.00 / 2
LoRA mean score: 0.50 / 2


In [63]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

# Create a separate directory for each evaluation run.
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path(CONFIG["output_dir"]) / timestamp
run_dir.mkdir(parents=True, exist_ok=False)

run_record = {
    "config": CONFIG,
    "base_model_revision": getattr(
        project_model.config, "_commit_hash", None
    ),
    "versions": {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "accelerate": accelerate.__version__,
    },
    "training_examples": train_examples,
    "validation": {
        "base_loss": base_loss,
        "lora_loss": lora_loss,
        "base_mean_human_score": base_mean,
        "lora_mean_human_score": lora_mean,
        "results": validation_results,
    },
}

results_path = run_dir / "experiment_results.json"

with results_path.open("w", encoding="utf-8") as file:
    json.dump(run_record, file, indent=2, ensure_ascii=False)

print("Saved experiment results:", results_path)

Saved experiment results: /content/lora-project/run-001/20261026T223206660604Z/experiment_results.json


In [64]:
# Save the trained LoRA adapter and tokenizer for inference.
adapter_dir = run_dir / "adapter"

project_model.save_pretrained(
    str(adapter_dir),
    safe_serialization=True,
)

tokenizer.save_pretrained(str(adapter_dir))

print("Adapter directory:", adapter_dir)

for path in sorted(adapter_dir.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / 1024:.1f} KiB")

Adapter directory: /content/lora-project/run-001/20261026T223206660604Z/adapter
README.md: 5.1 KiB
adapter_config.json: 1.1 KiB
adapter_model.safetensors: 4411.3 KiB
chat_template.jinja: 0.4 KiB
tokenizer.json: 3534.1 KiB
tokenizer_config.json: 0.4 KiB


In [65]:
from peft import PeftModel


reloaded_tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir))

reloaded_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    dtype=getattr(torch, CONFIG["dtype"]),
    device_map=0,
)

# Load the trained adapter for inference.
reloaded_model = PeftModel.from_pretrained(
    reloaded_base,
    str(adapter_dir),
    is_trainable=False,
)

prompt = validation_results[0]["prompt"]
saved_answer = validation_results[0]["lora_answer"]

reloaded_answer = generate_answer(
    reloaded_model,
    reloaded_tokenizer,
    prompt,
)

print("BEFORE SAVING:", saved_answer)
print("\nAFTER RELOADING:", reloaded_answer)
print("\nAnswers match:", saved_answer == reloaded_answer)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE SAVING: The low-rank factors are updated.

AFTER RELOADING: The low-rank factors are updated.

Answers match: True
